# 03 -- DueCare Agent Swarm Deep Dive (Generalized Framework)

**Technical-depth walkthrough of the 12-agent swarm and the
`AgentSupervisor` meta-agent.**

This notebook covers:

1. All 12 agents and what each contributes
2. The `AgentSupervisor` meta-agent (retry, budget, abort-on-harm)
3. The shared `AgentContext` blackboard pattern
4. A scripted walkthrough of data_generator -> anonymizer -> curator
5. Real PII redaction by the Anonymizer agent
6. A flaky agent retried by the Supervisor
7. The `harm_detected` abort pathway


## 1. Install and verify

In [ ]:
import subprocess, glob
for p in ['/kaggle/input/duecare-llm-wheels/*.whl', '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl', '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels: subprocess.check_call(['pip', 'install'] + wheels + ['--quiet'])

from duecare.agents import agent_registry, AgentSupervisor
from duecare.agents.base import SupervisorPolicy, BudgetExceeded, HarmDetected

print(f'Registered agents ({len(agent_registry)}):')
for agent_id in agent_registry.all_ids():
    agent = agent_registry.get(agent_id)
    print(f'  {agent_id:<22}  role={agent.role.value:<22}  budget=${agent.cost_budget_usd:.2f}')


## 2. The shared context blackboard

In [ ]:
from datetime import datetime
from duecare.core import AgentContext

ctx = AgentContext(
    run_id='demo_run_001',
    git_sha='abc123',
    workflow_id='demo',
    target_model_id='scripted:demo',
    domain_id='trafficking',
    started_at=datetime.now(),
)

# Agents read/write the context via record() and lookup()
ctx.record('initial_state', {'ready': True})
print(f'Context has: {list(ctx.outputs_by_agent.keys())}')
print(f'initial_state: {ctx.lookup("initial_state")}')


## 3. Data Generator -> Anonymizer -> Curator pipeline

In [ ]:
# DataGenerator emits 'synthetic_probes' to ctx
gen = agent_registry.get('data_generator')

# Seed synthetic probes with deliberately-PII-containing text
# to show the Anonymizer hard gate in action
ctx.record('synthetic_probes', [
    {'id': 'p001', 'text': 'Contact Maria at +1-555-0123 or maria@example.com'},
    {'id': 'p002', 'text': 'Her passport AB1234567 is being held'},
    {'id': 'p003', 'text': 'Transfer to IBAN DE89370400440532013000'},
    {'id': 'p004', 'text': 'Normal prompt with no PII.'},
])

# Anonymizer redacts PII via regex detection + verification
anon = agent_registry.get('anonymizer')
out = anon.execute(ctx)
print(f'Anonymizer decision: {out.decision}')
print(f'Anonymizer metrics:  {out.metrics}')

clean = ctx.lookup('clean_probes')
print(f'\nClean probes: {len(clean)}')
for p in clean:
    print(f'  {p["id"]}: {p["text"]}')


In [ ]:
# Curator dedupes and splits
cur = agent_registry.get('curator')
out = cur.execute(ctx)
print(f'Curator decision: {out.decision}')
print(f'Train: {len(ctx.lookup("train_jsonl"))}, Val: {len(ctx.lookup("val_jsonl"))}, Test: {len(ctx.lookup("test_jsonl"))}')


## 4. AgentSupervisor -- retry on transient failures

The supervisor wraps every agent call and retries on exceptions.
Below we build a flaky agent that fails its first two attempts
and succeeds on the third. The supervisor's retry policy catches
the first two failures and eventually surfaces the success.

In [ ]:
from duecare.core.enums import AgentRole, TaskStatus
from duecare.agents.base import fresh_agent_output

attempts = {'n': 0}

class FlakyAgent:
    id = 'flaky_demo'
    role = AgentRole.SCOUT
    version = '0.1.0'
    model = None
    tools = []
    inputs = set()
    outputs = {'ok'}
    cost_budget_usd = 0.0
    def execute(self, ctx):
        attempts['n'] += 1
        if attempts['n'] < 3:
            raise RuntimeError(f'transient failure #{attempts["n"]}')
        out = fresh_agent_output(self.id, self.role)
        out.status = TaskStatus.COMPLETED
        out.decision = f'succeeded on attempt {attempts["n"]}'
        return out
    def explain(self):
        return 'intentionally flaky'

supervisor = AgentSupervisor(SupervisorPolicy(
    max_retries=3,
    retry_backoff_s=0.01,
))

attempts['n'] = 0
output = supervisor.run(FlakyAgent(), ctx)
print(f'Final status: {output.status.value}')
print(f'Decision:     {output.decision}')
print(f'Total attempts: {attempts["n"]}')
print(f'Supervisor summary: {supervisor.summary()}')


## 5. AgentSupervisor -- abort on `harm_detected`

The Validator agent can signal that it found new harm in the
trained model by setting `ctx.record('harm_detected', True)`.
The supervisor checks this flag after every agent call and
raises `HarmDetected` -- aborting the whole workflow before the
Exporter can publish anything.

In [ ]:
class HarmDetectingAgent:
    id = 'harm_validator'
    role = AgentRole.VALIDATOR
    version = '0.1.0'
    model = None
    tools = []
    inputs = set()
    outputs = set()
    cost_budget_usd = 0.0
    def execute(self, ctx):
        ctx.record('harm_detected', True)
        out = fresh_agent_output(self.id, self.role)
        out.status = TaskStatus.COMPLETED
        out.decision = 'Found new harm in fine-tuned model'
        return out
    def explain(self):
        return 'harm detector'

abort_sup = AgentSupervisor(SupervisorPolicy(abort_on_harm=True))
ctx2 = AgentContext(
    run_id='harm_demo', git_sha='x', workflow_id='demo',
    target_model_id='m', domain_id='d', started_at=datetime.now(),
)

try:
    abort_sup.run(HarmDetectingAgent(), ctx2)
    print('Supervisor did NOT abort -- this is a bug')
except HarmDetected as e:
    print(f'HarmDetected raised as expected: {e}')
    print('The Exporter would never run after this.')


## 6. AgentSupervisor -- hard budget cap

In [ ]:
class ExpensiveAgent:
    id = 'expensive'
    role = AgentRole.DATA_GENERATOR
    version = '0.1.0'
    model = None
    tools = []
    inputs = set()
    outputs = set()
    cost_budget_usd = 0.6
    def execute(self, ctx):
        out = fresh_agent_output(self.id, self.role)
        out.status = TaskStatus.COMPLETED
        out.decision = 'generated probes'
        out.cost_usd = 0.6
        return out
    def explain(self): return 'costly'

sup = AgentSupervisor(SupervisorPolicy(hard_budget_usd=1.0))
ctx3 = AgentContext(
    run_id='budget_demo', git_sha='x', workflow_id='demo',
    target_model_id='m', domain_id='d', started_at=datetime.now(),
)

print(f'Call 1: total=$0.00 -> run')
sup.run(ExpensiveAgent(), ctx3)
print(f'Call 1 OK. total=${sup.total_cost_usd:.2f}')

print(f'\nCall 2: total=$0.60 -> run (under cap)')
sup.run(ExpensiveAgent(), ctx3)
print(f'Call 2 OK. total=${sup.total_cost_usd:.2f}')

print(f'\nCall 3: total=$1.20 (> $1.00 cap)')
try:
    sup.run(ExpensiveAgent(), ctx3)
    print('Did NOT abort - bug')
except BudgetExceeded as e:
    print(f'BudgetExceeded raised as expected: {e}')


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])
print('Plotly installed.')

## Agent swarm architecture visualization

In [ ]:
import plotly.graph_objects as go

# Build agent roster from the registry
agent_ids = list(agent_registry.all_ids())
agent_roles = []
agent_budgets = []
for aid in agent_ids:
    a = agent_registry.get(aid)
    agent_roles.append(a.role.value)
    agent_budgets.append(a.cost_budget_usd)

# Color-code by role category
role_colors = {
    'data_generator': '#636EFA',
    'anonymizer': '#EF553B',
    'curator': '#00CC96',
    'scout': '#AB63FA',
    'judge': '#FFA15A',
    'trainer': '#19D3F3',
    'validator': '#FF6692',
    'exporter': '#B6E880',
    'historian': '#FF97FF',
    'coordinator': '#FECB52',
}
colors = [role_colors.get(r, '#999999') for r in agent_roles]

fig = go.Figure(go.Bar(
    x=agent_budgets,
    y=agent_ids,
    orientation='h',
    marker_color=colors,
    text=[f'{r} -- ${b:.2f}' for r, b in zip(agent_roles, agent_budgets)],
    textposition='auto',
    hovertemplate='<b>%{y}</b><br>Role: %{text}<extra></extra>',
))

fig.update_layout(
    title='DueCare Agent Swarm -- All 12 Agents by Role and Budget',
    xaxis_title='Cost Budget (USD)',
    yaxis_title='Agent ID',
    height=500,
    margin=dict(l=160, r=40, t=60, b=40),
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
)
fig.show()

In [ ]:
import plotly.graph_objects as go

# Data flow funnel: raw probes -> anonymized -> curated -> evaluated -> reported
# These counts reflect what the pipeline demonstrated above.
# Raw probes seeded into the context.
raw_count = len(ctx.lookup('synthetic_probes')) if ctx.lookup('synthetic_probes') else 4
clean_count = len(ctx.lookup('clean_probes')) if ctx.lookup('clean_probes') else 0
train_count = len(ctx.lookup('train_jsonl')) if ctx.lookup('train_jsonl') else 0
val_count = len(ctx.lookup('val_jsonl')) if ctx.lookup('val_jsonl') else 0
test_count = len(ctx.lookup('test_jsonl')) if ctx.lookup('test_jsonl') else 0
curated_total = train_count + val_count + test_count

# Build the funnel stages
stages = ['Raw Probes', 'After Anonymizer', 'After Curator (total splits)', 'Train Split', 'Val Split', 'Test Split']
counts = [raw_count, clean_count, curated_total, train_count, val_count, test_count]

# Use a waterfall chart to show how items survive each pipeline stage
fig = go.Figure(go.Funnel(
    y=stages,
    x=counts,
    textinfo='value+percent initial',
    marker=dict(color=['#636EFA', '#EF553B', '#00CC96', '#19D3F3', '#AB63FA', '#FFA15A']),
    connector=dict(line=dict(color='#888', width=1)),
))

fig.update_layout(
    title='Data Flow Through the Agent Pipeline -- Items Surviving Each Stage',
    height=420,
    margin=dict(l=40, r=40, t=60, b=40),
    template='plotly_white',
)
fig.show()

## What this proves

This notebook demonstrates that all 12 agents register correctly on import via the shared `agent_registry`. The DataGenerator, Anonymizer, and Curator agents execute a real data flow pipeline where synthetic probes pass through each stage with full provenance tracking.

The Anonymizer agent performs genuine PII redaction. Phone numbers, email addresses, passport numbers, and IBANs are all detected and replaced with category tags such as `[PHONE]` and `[EMAIL]`, ensuring that no raw personally identifiable information survives into downstream stages.

The AgentSupervisor provides three critical safety mechanisms. First, it retries transient failures up to the configured `max_retries` limit before surfacing the error to the caller. Second, it checks the `harm_detected` flag after every agent execution and raises `HarmDetected` to abort the entire workflow before the Exporter can publish any artifacts. Third, it enforces a hard budget cap so that the workflow halts before cost overruns occur.

The interactive visualizations above show the full agent roster with their roles and budgets, and the funnel chart traces how many data items survive each pipeline stage from raw input through to curated train/val/test splits.

**Next:** [`04_submission_walkthrough.ipynb`](./04_submission_walkthrough.ipynb) -- the compact narrative attached to the Kaggle Writeup.